In [72]:
import numpy as np

In [73]:
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [74]:
train_embeddings = '/content/gdrive/MyDrive/Akib Bruh Mind Seer/embeddings/BERT embeddings/CLEF dataset/train_embeddings.npy'
train_labels = '/content/gdrive/MyDrive/Akib Bruh Mind Seer/embeddings/BERT embeddings/CLEF dataset/train_labels.npy'

test_embeddings = '/content/gdrive/MyDrive/Akib Bruh Mind Seer/embeddings/BERT embeddings/CLEF dataset/test_embeddings.npy'
test_labels = '/content/gdrive/MyDrive/Akib Bruh Mind Seer/embeddings/BERT embeddings/CLEF dataset/test_labels.npy'

In [75]:
train_X = np.load(train_embeddings)
train_Y = np.load(train_labels)

test_X = np.load(test_embeddings)
test_Y = np.load(test_labels)

In [76]:
print(train_X, train_Y)
print('~~~~~~')
print(test_X, test_Y)

[[ 0.11089098 -0.00446808  0.2494809  ... -0.04381233 -0.04551027
   0.05310011]
 [ 0.17040847 -0.00138917  0.26778182 ... -0.06879152 -0.06182676
  -0.09239165]
 [ 0.15571474 -0.00585976  0.1825483  ... -0.13049302 -0.02986942
  -0.05235168]
 ...
 [ 0.13473383 -0.04986885  0.21423917 ... -0.12040175  0.00394048
   0.05579036]
 [ 0.08177862 -0.13987777  0.40153483 ... -0.0371904  -0.08524916
  -0.1867226 ]
 [ 0.14360963 -0.00204951  0.1375092  ... -0.14928539 -0.04826211
  -0.0355715 ]] [0 0 0 0 1 0 0 1 1 1 1 0 0 0 1 0 0 0 0 0 1 1 0 0 0 1 0 0 0 0 0 0 0 0 1 1 0
 0 1 0 1 0 0 0 0 0 0 0 0 1 0 1 0 0 0 0 0 0 1 0 0 1 0 0 0 0 0 1 1 0 0 0 1 0
 1 0 1 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 1 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0
 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 1 0 0 0 0 0 0 0 0 1 0 0 0 0
 0 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0
 0 0 1 0 1 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 1 0 0 0 0 0 0

In [77]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(random_state=42, max_iter=1000)
clf = lr.fit(train_X, train_Y)
pred = lr.predict(test_X)


In [78]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import f1_score
from sklearn.metrics import classification_report

acc = accuracy_score(test_Y, pred)
print(f'Accuracy: {acc}')

conf_matrix = confusion_matrix(test_Y, pred)
print(f'Confusion Matrix: {conf_matrix}')

f1 = f1_score(test_Y, pred)
print(f'F1 Score: {f1}')


target_names = ['class 0', 'class 1']
clf_report = classification_report(test_Y, pred, target_names=target_names)
print(f'Classification Report\n {clf_report}')

tn, fp, fn, tp = confusion_matrix(test_Y, pred).ravel()

sensitivity = tp / (tp+fn)
specificity = tn / (tn+fp)

print(f'Sensitivity: {sensitivity}')
print(f'Specificity: {specificity}')

Accuracy: 0.908641975308642
Confusion Matrix: [[344   7]
 [ 30  24]]
F1 Score: 0.5647058823529412
Classification Report
               precision    recall  f1-score   support

     class 0       0.92      0.98      0.95       351
     class 1       0.77      0.44      0.56        54

    accuracy                           0.91       405
   macro avg       0.85      0.71      0.76       405
weighted avg       0.90      0.91      0.90       405

Sensitivity: 0.4444444444444444
Specificity: 0.98005698005698


### Keras stuff for fixing data imbalance https://keras.io/examples/structured_data/imbalanced_classification/

In [79]:
from sklearn.model_selection import train_test_split

# Creating validation set
# train_X = np.load('train_embeddings.npy')
# train_y = np.load('train_labels.npy')
X_train, X_val, y_train, y_val = train_test_split(train_X, train_Y, test_size=0.1, random_state=42)


# np.save('train_embeddings.npy', X_train)
# np.save('val_embeddings.npy', X_val)
# np.save('train_labels.npy', y_train)
# np.save('val_labels.npy', y_val)

In [91]:
## Analyze class imbalance in the targets. No need to Normalize

counts = np.bincount(y_train)
print(
    "Number of positive samples in training data: {} ({:.2f}% of total)".format(
        counts[1], 100 * float(counts[1]) / len(y_train)
    )
)

weight_for_0 = 1.0 / counts[0]
weight_for_1 = 1.0 / counts[1]

Number of positive samples in training data: 76 (17.43% of total)


In [92]:
## Build a binary classification model

from tensorflow import keras

model = keras.Sequential(
    [
        keras.layers.Dense(
            256, activation="relu", input_shape=(X_train.shape[-1],)
        ),
        keras.layers.Dense(256, activation="relu"),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(256, activation="relu"),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(1, activation="sigmoid"),
    ]
)
model.summary()

Model: "sequential_3"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
dense_12 (Dense)             (None, 256)               786688    
_________________________________________________________________
dense_13 (Dense)             (None, 256)               65792     
_________________________________________________________________
dropout_6 (Dropout)          (None, 256)               0         
_________________________________________________________________
dense_14 (Dense)             (None, 256)               65792     
_________________________________________________________________
dropout_7 (Dropout)          (None, 256)               0         
_________________________________________________________________
dense_15 (Dense)             (None, 1)                 257       
Total params: 918,529
Trainable params: 918,529
Non-trainable params: 0
________________________________________________

In [93]:
## Train the model with class_weight argument

metrics = [
    keras.metrics.FalseNegatives(name="fn"),
    keras.metrics.FalsePositives(name="fp"),
    keras.metrics.TrueNegatives(name="tn"),
    keras.metrics.TruePositives(name="tp"),
    keras.metrics.Precision(name="precision"),
    keras.metrics.Recall(name="recall"),
]

model.compile(
    optimizer=keras.optimizers.Adam(1e-2), loss="binary_crossentropy", metrics=metrics
)

callbacks = [keras.callbacks.ModelCheckpoint("fraud_model_at_epoch_{epoch}.h5")]
class_weight = {0: weight_for_0, 1: weight_for_1}

model.fit(
    X_train,
    y_train,
    batch_size=2048,
    epochs=30,
    verbose=2,
    callbacks=callbacks,
    validation_data=(X_val, y_val),
    class_weight=class_weight,
)

Epoch 1/30
1/1 - 3s - loss: 0.0034 - fn: 16.0000 - fp: 309.0000 - tn: 51.0000 - tp: 60.0000 - precision: 0.1626 - recall: 0.7895 - val_loss: 2.3559 - val_fn: 7.0000 - val_fp: 0.0000e+00 - val_tn: 42.0000 - val_tp: 0.0000e+00 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 2/30
1/1 - 0s - loss: 0.0375 - fn: 76.0000 - fp: 0.0000e+00 - tn: 360.0000 - tp: 0.0000e+00 - precision: 0.0000e+00 - recall: 0.0000e+00 - val_loss: 2.4423 - val_fn: 0.0000e+00 - val_fp: 42.0000 - val_tn: 0.0000e+00 - val_tp: 7.0000 - val_precision: 0.1429 - val_recall: 1.0000
Epoch 3/30
1/1 - 0s - loss: 0.0067 - fn: 0.0000e+00 - fp: 360.0000 - tn: 0.0000e+00 - tp: 76.0000 - precision: 0.1743 - recall: 1.0000 - val_loss: 0.4557 - val_fn: 7.0000 - val_fp: 0.0000e+00 - val_tn: 42.0000 - val_tp: 0.0000e+00 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 4/30
1/1 - 0s - loss: 0.0064 - fn: 76.0000 - fp: 1.0000 - tn: 359.0000 - tp: 0.0000e+00 - precision: 0.0000e+00 - recall: 0.0000e+00 - val_loss: 0.8

In [94]:
pred = model.predict(test_X)

In [97]:
pred = (pred > 0.5) # USE THIS OTHERWISE CONFUSION MATRIX WILL GIVE ERROR
print(accuracy_score(test_Y,pred))

0.7604938271604939


In [98]:
tn, fp, fn, tp = confusion_matrix(test_Y, pred).ravel()

sensitivity = tp / (tp+fn)
specificity = tn / (tn+fp)

print(f'Sensitivity: {sensitivity}')
print(f'Specificity: {specificity}')

Sensitivity: 0.8333333333333334
Specificity: 0.7492877492877493
